In [1]:
from google.colab import drive
drive.mount('/content/drive')
# rerun and get rid of unnecessary cells to determine contriever + better features
# plus better mlp effects

Mounted at /content/drive


In [ ]:
# One-time run, don't run again
%%bash
set -e
cd /content/drive/MyDrive
git clone -b mvp-logging-rewardhook https://github.com/Joseph2718/thesis.git

Cloning into 'thesis'...


In [10]:
# but run this every time you push new changes
%%bash
set -e
cd /content/drive/MyDrive/thesis
git fetch origin
git checkout mvp-logging-rewardhook
git reset --hard origin/mvp-logging-rewardhook
git clean -fd
git log -1 --oneline

M	scripts/set_utility_oracle_imitation.py
Your branch is behind 'origin/mvp-logging-rewardhook' by 1 commit, and can be fast-forwarded.
  (use "git pull" to update your local branch)
HEAD is now at 601abec contriever and better features
601abec contriever and better features


Already on 'mvp-logging-rewardhook'


In [13]:
%%bash
set -e
pip -q install datasets transformers sentencepiece accelerate faiss-cpu rank-bm25

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 110.4 MB/s eta 0:00:00


In [14]:
import torch, os, platform
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("CWD:", os.getcwd())

Torch: 2.10.0+cu128
CUDA available: True
GPU: NVIDIA H100 80GB HBM3
CWD: /content


In [ ]:
%%bash
set -e
cd /content/drive/MyDrive/thesis
ls -lh data/passage_corpus/wiki_passages_500k.jsonl || echo "missing corpus"
ls -lh data/retrieval_pools/hotpotqa_bm25_top20_train200_val80_seed42_wiki_passages_500k.jsonl || echo "missing pool"

-rw------- 1 root root 330M Feb 27 17:41 data/passage_corpus/wiki_passages_500k.jsonl
-rw------- 1 root root 3.7M Mar  3 20:15 data/retrieval_pools/hotpotqa_bm25_top20_train200_val80_seed42_wiki_passages_500k.jsonl


In [ ]:
%%bash
set -e
cd /content/thesis 2>/dev/null || cd /content/drive/MyDrive/thesis

# Download DPR wiki passages
wget -q https://dl.fbaipublicfiles.com/dpr/wikipedia_split/psgs_w100.tsv.gz
gunzip -f psgs_w100.tsv.gz

# Convert to jsonl subset (500k)
python scripts/prepare_wikipedia_passage_corpus.py \
  --input_corpus_path psgs_w100.tsv \
  --output_path data/passage_corpus/wiki_passages_500k.jsonl \
  --max_docs 500000

# Build retrieval pool
python scripts/build_retrieval_pool.py \
  --corpus_path data/passage_corpus/wiki_passages_500k.jsonl \
  --top_n 20 \
  --train_examples 200 \
  --val_examples 80 \
  --seed 42 \
  --output_path data/retrieval_pools/hotpotqa_bm25_top20_train200_val80_seed42_wiki_passages_500k.jsonl \
  --overwrite

Wrote 500000 normalized passages to data/passage_corpus/wiki_passages_500k.jsonl
Wrote retrieval pool to data/retrieval_pools/hotpotqa_bm25_top20_train200_val80_seed42_wiki_passages_500k.jsonl
Examples: 280, top_n: 20, corpus docs: 500000


Generating validation split: 100%|██████████| 7405/7405 [00:00<00:00, 42256.96 examples/s]


In [ ]:
%%bash
set -e
cd /content/drive/MyDrive/thesis

export POOL_PATH="data/retrieval_pools/hotpotqa_bm25_top20_train200_val80_seed42_wiki_passages_500k.jsonl"

python - <<'PY'
import hashlib, os
p = os.environ["POOL_PATH"]
h = hashlib.sha256()
with open(p, "rb") as f:
    for chunk in iter(lambda: f.read(1024*1024), b""):
        h.update(chunk)
print(h.hexdigest())
PY

464c22fdda6a351468f64ced274ee6cda4077d1ed7ebed67a82e3309a5680eda


In [ ]:
import os
os.chdir("/content/drive/MyDrive/thesis")

In [ ]:
%%bash
set -e
cd /content/drive/MyDrive/thesis

POOL="data/retrieval_pools/hotpotqa_bm25_top20_train200_val80_seed42_wiki_passages_500k.jsonl"

python scripts/set_utility_experiment.py \
  --candidate_pool_path "$POOL" \
  --singleton_cache_dir runs/singleton_cache \
  --compute_singletons 1 \
  --preflight_only 1 \
  --generator_model google/flan-t5-small \
  --utility_mode baseline_subtracted \
  --max_passage_tokens_for_prompt 64 \
  --max_input_len 512 \
  --max_target_len 64 \
  --k 5 \
  --train_examples 200 \
  --val_examples 80 \
  --data_seed 42 \
  --feature_mode dense \
  --feature_normalize per_query_zscore

Candidate pool SHA256: 464c22fdda6a351468f64ced274ee6cda4077d1ed7ebed67a82e3309a5680eda
--- Hardware ---
PyTorch generator device: cuda
CUDA available: True
CUDA device: NVIDIA H100 80GB HBM3
TF physical GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
----------------
Loading dense encoder: sentence-transformers/all-MiniLM-L6-v2 (device=cuda)
Encoder embedding dim: 384 -> feature dim: 385
Applied passage token cap for prompts: max_passage_tokens_for_prompt=64
Prompt passage chars (train): mean_raw=634.9 mean_capped=261.8 over 4000 passages
Prompt passage chars (val): mean_raw=634.7 mean_capped=261.9 over 1600 passages
Applied per_query_zscore normalization (eps=1e-06)
data_seed: 42 (data subset), seed: 42 (training randomness)
Feature mode: dense, input_dim: 386, feature_normalize: per_query_zscore, hidden_units: [64, 32], dropout: 0.00
train_ids SHA256: e077fef43ec849d2ad6e5b8e7fb3dae37dadad51379525b612831176505db4ee
val_ids SHA256:   02d6580e369df2cb245520d27

2026-03-03 23:29:09.302526: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-03 23:29:09.315285: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772580549.330682   23333 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772580549.335543   23333 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772580549.346934   23333 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [ ]:
POOL = "/content/drive/MyDrive/thesis/data/retrieval_pools/hotpotqa_bm25_top20_train200_val80_seed42_wiki_passages_500k.jsonl"
GEN = "google/flan-t5-small"
CAP = 64
MAX_INPUT_LEN = 512
MAX_TARGET_LEN = 64
K = 5

In [ ]:
import json, numpy as np, torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(GEN)
gen = AutoModelForSeq2SeqLM.from_pretrained(GEN).to(device).eval()

EMPTY="<none>"

def build_prompt(q, passages):
    ctx = "\n".join([f"[{i+1}] {p}" for i,p in enumerate(passages)]) if passages else EMPTY
    return f"question: {q}\ncontext:\n{ctx}\nanswer:"

@torch.no_grad()
def neg_nll(q, passages, gold):
    x = tokenizer(build_prompt(q, passages), return_tensors="pt",
                  truncation=True, max_length=MAX_INPUT_LEN).to(device)
    labels = tokenizer(gold, return_tensors="pt",
                       truncation=True, max_length=MAX_TARGET_LEN,
                       padding="max_length").input_ids.to(device)
    labels = labels.masked_fill(labels == tokenizer.pad_token_id, -100)
    out = gen(**x, labels=labels)
    return float(-out.loss.item())

baseline_cache = {}
def utility(q, passages, gold, key):
    if key not in baseline_cache:
        baseline_cache[key] = neg_nll(q, [], gold)
    return neg_nll(q, passages, gold) - baseline_cache[key]

# Load pool rows
rows=[]
with open(POOL,"r") as f:
    for line in f:
        if line.strip():
            rows.append(json.loads(line))

# Take validation rows
val=[r for r in rows if str(r.get("split","")).startswith("val") or str(r.get("example_id","")).startswith("val_")]
NQ=40
val = val[:NQ] if len(val)>=NQ else rows[-NQ:]

def oracle_singleton_topk(q, passages, gold, key):
    us=[utility(q,[p],gold,key) for p in passages]
    idx=np.argsort(-np.array(us))[:K]
    sel=[passages[i] for i in idx]
    return utility(q, sel, gold, key)

def oracle_greedy(q, passages, gold, key):
    chosen=[]
    remaining=list(range(len(passages)))
    cur=utility(q,[],gold,key)
    for _ in range(min(K,len(passages))):
        best_i, best_u=None, None
        for i in remaining:
            trial=[passages[j] for j in chosen+[i]]
            u=utility(q, trial, gold, key)
            if best_u is None or u>best_u:
                best_u=u; best_i=i
        chosen.append(best_i)
        remaining.remove(best_i)
        cur=best_u
    return cur

out=[]
for r in val:
    q=r["question"]
    gold=r.get("gold_answer", r.get("answer",""))
    passages = r.get("passages_for_prompt", r["passages"])
    key=str(r.get("example_id",""))
    base=utility(q, passages[:K], gold, key)
    osing=oracle_singleton_topk(q, passages, gold, key)
    ogreedy=oracle_greedy(q, passages, gold, key)
    out.append((base, osing, ogreedy))

out=np.array(out, dtype=np.float64)
print(f"Headroom check on {len(out)} val queries (K={K}, cap={CAP})")
print("Baseline mean:", out[:,0].mean(), "std:", out[:,0].std())
print("Oracle singleton-topK mean:", out[:,1].mean(), "std:", out[:,1].std())
print("Oracle greedy mean:", out[:,2].mean(), "std:", out[:,2].std())
print("Mean (oracle_singleton - baseline):", (out[:,1]-out[:,0]).mean())
print("Mean (oracle_greedy - baseline):   ", (out[:,2]-out[:,0]).mean())

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Headroom check on 40 val queries (K=5, cap=64)
Baseline mean: 0.002123308181762695 std: 0.12460924383174975
Oracle singleton-topK mean: 0.03681941032409668 std: 0.13796199690336794
Oracle greedy mean: 0.0948251485824585 std: 0.13757554690079743
Mean (oracle_singleton - baseline): 0.034696102142333984
Mean (oracle_greedy - baseline):    0.0927018404006958


## Phase 1: Oracle imitation with flan-t5-base
Check if a stronger generator produces a sharper utility signal that the scorer can learn.

In [ ]:
%%bash
set -e
cd /content/drive/MyDrive/thesis

python scripts/set_utility_oracle_imitation.py \
  --candidate_pool_path "data/retrieval_pools/hotpotqa_bm25_top20_train200_val80_seed42_wiki_passages_500k.jsonl" \
  --expected_pool_sha256 "464c22fdda6a351468f64ced274ee6cda4077d1ed7ebed67a82e3309a5680eda" \
  --generator_model google/flan-t5-base \
  --k 5 \
  --max_passages 20 \
  --max_passage_tokens_for_prompt 64 \
  --max_input_len 512 \
  --max_target_len 64 \
  --feature_mode dense \
  --feature_normalize per_query_zscore \
  --supervised_loss listwise \
  --minibatch_queries 16 \
  --max_steps 300 \
  --eval_every 20 \
  --singleton_cache_dir runs/singleton_cache \
  --compute_singletons 1 \
  --output_dir "runs/oracle_imitation/contriever_flan_t5_base"

## Phase 1: PG vs PL-Rank with flan-t5-base
Run only if oracle imitation above shows clear improvement over baseline.

In [ ]:
%%bash
set -e
cd /content/drive/MyDrive/thesis

POOL="data/retrieval_pools/hotpotqa_bm25_top20_train200_val80_seed42_wiki_passages_500k.jsonl"
HASH="464c22fdda6a351468f64ced274ee6cda4077d1ed7ebed67a82e3309a5680eda"

python scripts/set_utility_experiment.py \
  --candidate_pool_path "$POOL" \
  --expected_pool_sha256 "$HASH" \
  --generator_model google/flan-t5-base \
  --train_examples 200 \
  --val_examples 80 \
  --max_passages 20 \
  --max_passage_tokens_for_prompt 64 \
  --max_input_len 512 \
  --max_target_len 64 \
  --k 5 \
  --feature_mode dense \
  --feature_normalize per_query_zscore \
  --num_samples 16 \
  --minibatch_queries 8 \
  --max_steps 200 \
  --eval_every 20 \
  --learning_rate 0.01 \
  --data_seed 42 \
  --seed 41 \
  --singleton_cache_dir runs/singleton_cache \
  --compute_singletons 0 \
  --output_dir "runs/pg_vs_plrank_contriever_t5base_s41"

## Phase 2: More training data (500 train / 150 val)
Rebuild retrieval pool with more queries, then rerun oracle + PG vs PL-Rank. Only run if Phase 1 still flat.

In [ ]:
%%bash
set -e
cd /content/drive/MyDrive/thesis

python scripts/build_retrieval_pool.py \
  --corpus_path data/passage_corpus/wiki_passages_500k.jsonl \
  --top_n 20 \
  --train_examples 500 \
  --val_examples 150 \
  --seed 42 \
  --output_path data/retrieval_pools/hotpotqa_bm25_top20_train500_val150_seed42_wiki_passages_500k.jsonl \
  --overwrite

In [ ]:
%%bash
set -e
cd /content/drive/MyDrive/thesis

POOL="data/retrieval_pools/hotpotqa_bm25_top20_train500_val150_seed42_wiki_passages_500k.jsonl"

# Get hash of new pool
HASH=$(python3 -c "
import hashlib
h = hashlib.sha256()
with open('$POOL', 'rb') as f:
    for chunk in iter(lambda: f.read(1024*1024), b''):
        h.update(chunk)
print(h.hexdigest())
")
echo "New pool hash: $HASH"

# Oracle imitation with flan-t5-base + larger pool
python scripts/set_utility_oracle_imitation.py \
  --candidate_pool_path "$POOL" \
  --expected_pool_sha256 "$HASH" \
  --generator_model google/flan-t5-base \
  --train_examples 500 \
  --val_examples 150 \
  --k 5 \
  --max_passages 20 \
  --max_passage_tokens_for_prompt 64 \
  --max_input_len 512 \
  --max_target_len 64 \
  --feature_mode dense \
  --feature_normalize per_query_zscore \
  --supervised_loss listwise \
  --minibatch_queries 16 \
  --max_steps 300 \
  --eval_every 20 \
  --singleton_cache_dir runs/singleton_cache \
  --compute_singletons 1 \
  --output_dir "runs/oracle_imitation/contriever_t5base_500train"

In [ ]:
%%bash
set -e
cd /content/drive/MyDrive/thesis

POOL="data/retrieval_pools/hotpotqa_bm25_top20_train500_val150_seed42_wiki_passages_500k.jsonl"
HASH=$(python3 -c "
import hashlib
h = hashlib.sha256()
with open('$POOL', 'rb') as f:
    for chunk in iter(lambda: f.read(1024*1024), b''):
        h.update(chunk)
print(h.hexdigest())
")

python scripts/set_utility_experiment.py \
  --candidate_pool_path "$POOL" \
  --expected_pool_sha256 "$HASH" \
  --generator_model google/flan-t5-base \
  --train_examples 500 \
  --val_examples 150 \
  --max_passages 20 \
  --max_passage_tokens_for_prompt 64 \
  --max_input_len 512 \
  --max_target_len 64 \
  --k 5 \
  --feature_mode dense \
  --feature_normalize per_query_zscore \
  --num_samples 16 \
  --minibatch_queries 8 \
  --max_steps 200 \
  --eval_every 20 \
  --learning_rate 0.01 \
  --data_seed 42 \
  --seed 41 \
  --singleton_cache_dir runs/singleton_cache \
  --compute_singletons 1 \
  --output_dir "runs/pg_vs_plrank_t5base_500train_s41"

## Contriever Retrieval Pool + Oracle + PG vs PL-Rank

In [ ]:
%%bash
set -e
cd /content/drive/MyDrive/thesis

python scripts/build_retrieval_pool.py \
  --corpus_path data/passage_corpus/wiki_passages_500k.jsonl \
  --retriever contriever \
  --top_n 20 \
  --train_examples 500 \
  --val_examples 150 \
  --seed 42 \
  --overwrite

In [ ]:
%%bash
set -e
cd /content/drive/MyDrive/thesis

POOL="data/retrieval_pools/hotpotqa_contriever_top20_train500_val150_seed42_wiki_passages_500k.jsonl"
HASH=$(python3 -c "
import hashlib
h = hashlib.sha256()
with open('$POOL', 'rb') as f:
    for chunk in iter(lambda: f.read(1024*1024), b''):
        h.update(chunk)
print(h.hexdigest())
")
echo "Contriever pool hash: $HASH"

python scripts/set_utility_oracle_imitation.py \
  --candidate_pool_path "$POOL" \
  --expected_pool_sha256 "$HASH" \
  --generator_model google/flan-t5-base \
  --train_examples 500 \
  --val_examples 150 \
  --k 5 \
  --max_passages 20 \
  --max_passage_tokens_for_prompt 64 \
  --max_input_len 512 \
  --max_target_len 64 \
  --feature_mode dense \
  --feature_normalize per_query_zscore \
  --supervised_loss listwise \
  --minibatch_queries 16 \
  --max_steps 300 \
  --eval_every 20 \
  --singleton_cache_dir runs/singleton_cache \
  --compute_singletons 1 \
  --output_dir "runs/oracle_imitation/contriever_retrieval_t5base_500train"

In [ ]:
%%bash
set -e
cd /content/drive/MyDrive/thesis

POOL="data/retrieval_pools/hotpotqa_contriever_top20_train500_val150_seed42_wiki_passages_500k.jsonl"
HASH=$(python3 -c "
import hashlib
h = hashlib.sha256()
with open('$POOL', 'rb') as f:
    for chunk in iter(lambda: f.read(1024*1024), b''):
        h.update(chunk)
print(h.hexdigest())
")

python scripts/set_utility_experiment.py \
  --candidate_pool_path "$POOL" \
  --expected_pool_sha256 "$HASH" \
  --generator_model google/flan-t5-base \
  --train_examples 500 \
  --val_examples 150 \
  --max_passages 20 \
  --max_passage_tokens_for_prompt 64 \
  --max_input_len 512 \
  --max_target_len 64 \
  --k 5 \
  --feature_mode dense \
  --feature_normalize per_query_zscore \
  --num_samples 16 \
  --minibatch_queries 8 \
  --max_steps 200 \
  --eval_every 20 \
  --learning_rate 0.01 \
  --data_seed 42 \
  --seed 42 \
  --singleton_cache_dir runs/singleton_cache \
  --compute_singletons 1 \
  --output_dir "runs/pg_vs_plrank_contriever_t5base_500train"

In [ ]:
%%bash
set -e
cd /content/drive/MyDrive/thesis

POOL="data/retrieval_pools/hotpotqa_contriever_top20_train500_val150_seed42_wiki_passages_500k.jsonl"
HASH=$(python3 -c "
import hashlib
h = hashlib.sha256()
with open('$POOL', 'rb') as f:
    for chunk in iter(lambda: f.read(1024*1024), b''):
        h.update(chunk)
print(h.hexdigest())
")

python scripts/set_utility_experiment.py \
  --candidate_pool_path "$POOL" \
  --expected_pool_sha256 "$HASH" \
  --generator_model google/flan-t5-base \
  --train_examples 500 \
  --val_examples 150 \
  --max_passages 20 \
  --max_passage_tokens_for_prompt 64 \
  --max_input_len 512 \
  --max_target_len 64 \
  --k 5 \
  --feature_mode dense \
  --feature_normalize per_query_zscore \
  --num_samples 16 \
  --minibatch_queries 8 \
  --max_steps 200 \
  --eval_every 20 \
  --learning_rate 0.01 \
  --data_seed 42 \
  --seed 43 \
  --singleton_cache_dir runs/singleton_cache \
  --compute_singletons 0 \
  --output_dir "runs/pg_vs_plrank_contriever_t5base_500train_s43"

In [ ]:
%%bash
set -e
cd /content/drive/MyDrive/thesis

POOL="data/retrieval_pools/hotpotqa_contriever_top20_train500_val150_seed42_wiki_passages_500k.jsonl"
HASH=$(python3 -c "
import hashlib
h = hashlib.sha256()
with open('$POOL', 'rb') as f:
    for chunk in iter(lambda: f.read(1024*1024), b''):
        h.update(chunk)
print(h.hexdigest())
")

python scripts/set_utility_experiment.py \
  --candidate_pool_path "$POOL" \
  --expected_pool_sha256 "$HASH" \
  --generator_model google/flan-t5-base \
  --train_examples 500 \
  --val_examples 150 \
  --max_passages 20 \
  --max_passage_tokens_for_prompt 64 \
  --max_input_len 512 \
  --max_target_len 64 \
  --k 5 \
  --feature_mode dense \
  --feature_normalize per_query_zscore \
  --num_samples 16 \
  --minibatch_queries 8 \
  --max_steps 200 \
  --eval_every 20 \
  --learning_rate 0.01 \
  --data_seed 42 \
  --seed 41 \
  --singleton_cache_dir runs/singleton_cache \
  --compute_singletons 0 \
  --output_dir "runs/pg_vs_plrank_contriever_t5base_500train_s41"

## Phase 3: LR sweep
If oracle looks good but RL is still flat, try lower learning rates and more steps. Uses whichever pool/generator worked best above.

In [ ]:
%%bash
set -e
cd /content/drive/MyDrive/thesis

# Adjust POOL/HASH to whichever pool worked best in earlier phases
POOL="data/retrieval_pools/hotpotqa_bm25_top20_train500_val150_seed42_wiki_passages_500k.jsonl"
HASH=$(python3 -c "
import hashlib
h = hashlib.sha256()
with open('$POOL', 'rb') as f:
    for chunk in iter(lambda: f.read(1024*1024), b''):
        h.update(chunk)
print(h.hexdigest())
")

for LR in 0.003 0.001; do
  echo "===== LR=$LR ====="
  python scripts/set_utility_experiment.py \
    --candidate_pool_path "$POOL" \
    --expected_pool_sha256 "$HASH" \
    --generator_model google/flan-t5-base \
    --train_examples 500 \
    --val_examples 150 \
    --max_passages 20 \
    --max_passage_tokens_for_prompt 64 \
    --max_input_len 512 \
    --max_target_len 64 \
    --k 5 \
    --feature_mode dense \
    --feature_normalize per_query_zscore \
    --num_samples 32 \
    --minibatch_queries 8 \
    --max_steps 500 \
    --eval_every 50 \
    --learning_rate "$LR" \
    --data_seed 42 \
    --seed 41 \
    --singleton_cache_dir runs/singleton_cache \
    --compute_singletons 0 \
    --output_dir "runs/lr_sweep/t5base_500train_lr${LR}_ns32_s41"
done

## Phase 4: Larger candidate pool (top_n=50)
Last resort. Rebuild pool with 50 passages per query to give the scorer more room to differentiate.

In [ ]:
%%bash
set -e
cd /content/drive/MyDrive/thesis

# Rebuild pool with top_n=50
python scripts/build_retrieval_pool.py \
  --corpus_path data/passage_corpus/wiki_passages_500k.jsonl \
  --top_n 50 \
  --train_examples 500 \
  --val_examples 150 \
  --seed 42 \
  --output_path data/retrieval_pools/hotpotqa_bm25_top50_train500_val150_seed42_wiki_passages_500k.jsonl \
  --overwrite

In [ ]:
%%bash
set -e
cd /content/drive/MyDrive/thesis

POOL="data/retrieval_pools/hotpotqa_bm25_top50_train500_val150_seed42_wiki_passages_500k.jsonl"
HASH=$(python3 -c "
import hashlib
h = hashlib.sha256()
with open('$POOL', 'rb') as f:
    for chunk in iter(lambda: f.read(1024*1024), b''):
        h.update(chunk)
print(h.hexdigest())
")

# Oracle imitation first
python scripts/set_utility_oracle_imitation.py \
  --candidate_pool_path "$POOL" \
  --expected_pool_sha256 "$HASH" \
  --generator_model google/flan-t5-base \
  --k 5 \
  --max_passages 50 \
  --max_passage_tokens_for_prompt 64 \
  --max_input_len 512 \
  --max_target_len 64 \
  --feature_mode dense \
  --feature_normalize per_query_zscore \
  --supervised_loss listwise \
  --minibatch_queries 16 \
  --max_steps 300 \
  --eval_every 20 \
  --singleton_cache_dir runs/singleton_cache \
  --compute_singletons 1 \
  --output_dir "runs/oracle_imitation/contriever_t5base_500train_top50"

In [ ]:
%%bash
set -e
cd /content/drive/MyDrive/thesis

POOL="data/retrieval_pools/hotpotqa_bm25_top50_train500_val150_seed42_wiki_passages_500k.jsonl"
HASH=$(python3 -c "
import hashlib
h = hashlib.sha256()
with open('$POOL', 'rb') as f:
    for chunk in iter(lambda: f.read(1024*1024), b''):
        h.update(chunk)
print(h.hexdigest())
")

python scripts/set_utility_experiment.py \
  --candidate_pool_path "$POOL" \
  --expected_pool_sha256 "$HASH" \
  --generator_model google/flan-t5-base \
  --train_examples 500 \
  --val_examples 150 \
  --max_passages 50 \
  --max_passage_tokens_for_prompt 64 \
  --max_input_len 512 \
  --max_target_len 64 \
  --k 5 \
  --feature_mode dense \
  --feature_normalize per_query_zscore \
  --num_samples 16 \
  --minibatch_queries 8 \
  --max_steps 500 \
  --eval_every 50 \
  --learning_rate 0.003 \
  --data_seed 42 \
  --seed 41 \
  --singleton_cache_dir runs/singleton_cache \
  --compute_singletons 0 \
  --output_dir "runs/pg_vs_plrank_t5base_500train_top50_s41"